In [ ]:
import requests
import re
import json

OLLAMA_HOST = "http://192.168.86.100:11434"
DEEPSEEK_MODEL = "deepseek-r1"
QWEN_MODEL = "qwen3"
OLLAMA_TIMEOUT = 300

session = requests.Session()
session.trust_env = False  # 🔥 IMPORTANT

def query_ollama(prompt: str, model:str):
    url = f"{OLLAMA_HOST}/api/chat"

    payload = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "stream": False,
        "options": {
            "temperature": 0
        }
    }

    response = session.post(url, json=payload, timeout=OLLAMA_TIMEOUT)
    response.raise_for_status()

    content = response.json()["message"]["content"]
    return split_reasoning_and_answer(content)

def split_reasoning_and_answer(text: str):
    match = re.search(r"<think>(.*?)</think>", text, re.DOTALL)
    if match:
        reasoning = match.group(1).strip()
        answer = text.replace(match.group(0), "").strip()
    else:
        reasoning = ""
        answer = text.strip()
    return reasoning, answer

def clean_and_parse_json(json_str: str) -> dict:
    """
    Cleans common JSON formatting issues and parses into a dict.
    Handles:
    - Markdown code fences
    - Smart quotes
    - Trailing commas
    """

    # Remove Markdown code fences
    json_str = re.sub(r"```(?:json)?\s*", "", json_str)
    json_str = re.sub(r"\s*```", "", json_str)

    # Replace smart quotes with normal quotes
    json_str = json_str.replace("“", '"').replace("”", '"')
    json_str = json_str.replace("‘", "'").replace("’", "'")

    # Remove trailing commas before } or ]
    json_str = re.sub(r",\s*([}\]])", r"\1", json_str)

    return json.loads(json_str)

def to_camel_case(value: str) -> str:
    """
    Convert a name string to Camel Case.
    Example: 'joHN doe' -> 'John Doe'
    """
    if not value or not isinstance(value, str):
        return ""

    parts = re.split(r"\s+", value.strip())
    result = " ".join(p.capitalize() for p in parts)
    return result.strip()

def merge_name_json_strings(json_str_a: str, json_str_b: str) -> dict:
    """
    Merge two JSON strings containing first_name and last_name.
    A field is kept only if both JSON objects agree on the same value.
    Otherwise, the field is set to an empty string.
    """

    try:
        a = clean_and_parse_json(json_str_a)
        b = clean_and_parse_json(json_str_b)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON input: {e} {json_str_a} {json_str_b}")

    if not isinstance(a, dict) or not isinstance(b, dict):
        raise ValueError("JSON inputs must represent objects")

    result = {}
    result["first_name"] = ""
    result["last_name"] = ""

    #if any of the result is not confident about any name, then return empty name
    if a.get("first_name") == "" and a.get("last_name") == "" :
        return result
    if b.get("first_name") == "" and b.get("last_name") == "" :
        return result

    #merge the names from both results
    for field in ("first_name", "last_name"):
        val_a = to_camel_case(a.get(field, ""))
        val_b = to_camel_case(b.get(field, ""))

        if val_a and not val_b:
            result[field] = val_a
        elif val_b and not val_a:
            result[field] = val_b
        elif val_a and val_b and val_a == val_b:
            result[field] = val_a
        else:
            result[field] = ""

    return result

def infer_name(email: str):
    email_id = email.split("@")[0]
    print(email_id)
    prompt = f"From email user ID {email_id}, infer the most likely last name that you are highly confident with. If you are confident with the last name, then proceed to infer the first name that you are highly confident with. If you are not sure, then leave first name and last name as empty string. Return the final result in json format with key 'first_name' and 'last_name'."
    reasoning_deepseek, answer_deepseek = query_ollama(prompt, DEEPSEEK_MODEL)
    reasoning_qwen, answer_qwen = query_ollama(prompt, QWEN_MODEL)
    final_answer = merge_name_json_strings(answer_deepseek, answer_qwen)
    return final_answer


In [ ]:
import pandas as pd

# Load CSV Data
csv_file = 'ACEC_CA_Members.csv'
df = pd.read_csv(csv_file)

for index, row in df.iterrows():
        try:
            # Email
            email = row['email']
            if pd.isnull(email):
                continue
            answer = infer_name(email)
            
            print("Final answer: [", answer['first_name'], "] [", answer['last_name'], "]")
            df.at[index, 'contact_first_name'] = answer['first_name']
            df.at[index, 'contact_last_name'] = answer['last_name']

        except Exception as e:
            print(f"Error processing row {index}: {str(e)}")

# Save updated DataFrame
df.to_csv('ACEC_CA_Members_updated.csv', index=False)

In [ ]:
import pandas as pd
import re

def parse_address_heuristic(addr):
    if not isinstance(addr, str) or not addr.strip():
        return {"street": "", "city": "", "state": "", "zip": ""}
    
    # Split by comma
    parts = [p.strip() for p in addr.split(',')]
    
    # Heuristic assumption: [Street...], [City], [State], [Zip]
    
    if len(parts) < 2:
        return {"street": addr, "city": "", "state": "", "zip": ""}

    zip_code = ""
    state = ""
    city = ""
    street = ""

    # Check last part for Zip
    last_part = parts[-1]
    # Simple check for US zip (5 digits or 5-4)
    if re.match(r'^\d{5}(-\d{4})?$', last_part):
        zip_code = last_part
        if len(parts) >= 2:
            state = parts[-2]
        if len(parts) >= 3:
            city = parts[-3]
        if len(parts) >= 4:
            street = ", ".join(parts[:-3])
    else:
        # Assume no zip, last part is state
        state = last_part
        if len(parts) >= 2:
            city = parts[-2]
        if len(parts) >= 3:
            street = ", ".join(parts[:-2])
            
    return {"street": street, "city": city, "state": state, "zip": zip_code}

def normalize_for_comparison(s):
    # Remove spaces and commas, lower case for loose verification
    if not isinstance(s, str): return ""
    return re.sub(r'[\s,]', '', s.lower())

csv_file = 'ACEC_CA_Members_updated.csv'
df = pd.read_csv(csv_file)

# Add new columns if they don't exist
for col in ['street', 'city', 'state', 'zip_code']:
    if col not in df.columns:
        df[col] = ""

processed_count = 0
for index, row in df.iterrows():
    addr = row['address']
    if pd.isna(addr) or not str(addr).strip():
        continue
        
    parsed = parse_address_heuristic(str(addr))
    
    # Reconstruct for verification
    parts_to_join = []
    if parsed['street']: parts_to_join.append(parsed['street'])
    if parsed['city']: parts_to_join.append(parsed['city'])
    if parsed['state']: parts_to_join.append(parsed['state'])
    if parsed['zip']: parts_to_join.append(parsed['zip'])
    
    reconstructed = ", ".join(parts_to_join)
    
    # Loose match verification
    if normalize_for_comparison(reconstructed) != normalize_for_comparison(addr):
        print(f"Mismatch at row {index}: '{addr}' vs '{reconstructed}'")
        # Use Ollama for correction if mismatch
        try:
            prompt = f"Parse the following address into JSON with keys 'street', 'city', 'state', 'zip'. Address: {addr}"
            reasoning, answer = query_ollama(prompt, QWEN_MODEL)
            llm_parsed = clean_and_parse_json(answer)
            
            parsed['street'] = llm_parsed.get('street', '')
            parsed['city'] = llm_parsed.get('city', '')
            parsed['state'] = llm_parsed.get('state', '')
            parsed['zip'] = llm_parsed.get('zip', '')
            print(f"LLM corrected to: {parsed}")
        except Exception as e:
            print(f"LLM failed: {e}")

    df.at[index, 'street'] = parsed['street']
    df.at[index, 'city'] = parsed['city']
    df.at[index, 'state'] = parsed['state']
    df.at[index, 'zip_code'] = parsed['zip']
    processed_count += 1

df.to_csv('ACEC_CA_Members_updated.csv', index=False)
print(f"Address separation complete. Processed {processed_count} addresses.")



In [5]:
import pandas as pd
import psycopg2
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Database connection parameters
DB_HOST = os.getenv('DB_HOST', '192.168.86.100')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'aquiferpe')
DB_USER = os.getenv('DB_USER', 'aquifer_app')
DB_PASSWORD = os.getenv('DB_PASSWORD')

def get_db_connection():
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )
    return conn

# Load updated CSV
df = pd.read_csv('ACEC_CA_Members_updated.csv')

try:
    conn = get_db_connection()
    cur = conn.cursor()
    
    clients_added = 0
    warnings = 0
    errors = 0

    print("Starting client import...")

    for index, row in df.iterrows():
        try:
            # Check for non-empty name
            first_name = row.get('contact_first_name', '')
            last_name = row.get('contact_last_name', '')
            
            # Ensure they are strings
            if pd.isna(first_name): first_name = ''
            if pd.isna(last_name): last_name = ''
            
            # Check if at least first name or last name is not empty
            if not str(first_name).strip() and not str(last_name).strip():
                continue

            email = row.get('email')
            if pd.isna(email) or not str(email).strip():
                continue
            
            # Check uniqueness of email
            cur.execute("SELECT id FROM client WHERE email = %s", (email,))
            if cur.fetchone():
                # Already exists
                continue

            # Find company_id
            company_name = row.get('name')
            company_id = None
            if not pd.isna(company_name):
                cur.execute("SELECT id FROM company WHERE company_name = %s", (company_name,))
                res = cur.fetchone()
                if res:
                    company_id = res[0]
                else:
                    print(f"Warning: Company '{company_name}' not found for client {email}. Continuing...")
                    warnings += 1
            
            # Insert client
            cur.execute("""
                INSERT INTO client (first_name, last_name, email, company_id)
                VALUES (%s, %s, %s, %s)
            """, (str(first_name).strip(), str(last_name).strip(), str(email).strip(), company_id))
            
            clients_added += 1
            print(f"Added client {first_name} {last_name} {email} {company_name}")
            
        except Exception as e:
            print(f"Error processing row {index}: {e}")
            errors += 1
            conn.rollback()

    conn.commit()
    print(f"Finished. Clients added: {clients_added}, Warnings: {warnings}, Errors: {errors}")

except Exception as e:
    print(f"Critical Error: {e}")
finally:
    if 'cur' in locals(): cur.close()
    if 'conn' in locals(): conn.close()


Starting client import...
Finished. Clients added: 0, Warnings: 0, Errors: 0


In [ ]:
import pandas as pd
import psycopg2
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Database connection parameters
DB_HOST = os.getenv('DB_HOST', '192.168.86.100')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'aquiferpe')
DB_USER = os.getenv('DB_USER', 'aquifer_app')
DB_PASSWORD = os.getenv('DB_PASSWORD')

def get_db_connection():
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )
    return conn

# Load updated CSV
df = pd.read_csv('ACEC_CA_Members_updated.csv')

try:
    conn = get_db_connection()
    cur = conn.cursor()
    
    updated_count = 0
    not_found_count = 0
    errors = 0

    print("Starting company address update...")

    for index, row in df.iterrows():
        try:
            company_name = row.get('name')
            if pd.isna(company_name) or not str(company_name).strip():
                continue

            # Check if company exists
            cur.execute("SELECT id FROM company WHERE company_name = %s", (company_name,))
            res = cur.fetchone()
            
            if res:
                company_id = res[0]
                
                # Get address components
                street = row.get('street', '')
                city = row.get('city', '')
                state = row.get('state', '')
                zip_code = row.get('zip_code', '')
                
                # Ensure values are strings (handle NaN)
                if pd.isna(street): street = ''
                if pd.isna(city): city = ''
                if pd.isna(state): state = ''
                if pd.isna(zip_code): zip_code = ''
                
                # Update company record
                update_query = """
                    UPDATE company 
                    SET street = %s, city = %s, state = %s, zip_code = %s, updated_at = CURRENT_TIMESTAMP
                    WHERE id = %s
                """
                cur.execute(update_query, (str(street).strip(), str(city).strip(), str(state).strip(), str(zip_code).strip(), company_id))
                
                updated_count += 1
            else:
                not_found_count += 1
            
        except Exception as e:
            print(f"Error processing row {index}: {e}")
            errors += 1
            conn.rollback()

    conn.commit()
    print(f"Finished. Updated: {updated_count}, Not Found: {not_found_count}, Errors: {errors}")

except Exception as e:
    print(f"Critical Error: {e}")
finally:
    if 'cur' in locals(): cur.close()
    if 'conn' in locals(): conn.close()



In [ ]:
import pandas as pd
import psycopg2
import os
from dotenv import load_dotenv
import sys

# Load environment variables
load_dotenv()

# Database connection parameters
DB_HOST = os.getenv('DB_HOST', '192.168.86.100')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'aquiferpe')
DB_USER = os.getenv('DB_USER', 'aquifer_app')
DB_PASSWORD = os.getenv('DB_PASSWORD')

def get_db_connection():
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )
    return conn

# Load updated CSV
print("Loading CSV file...")
df = pd.read_csv('ACEC_CA_Members_updated.csv')
print(f"Loaded {len(df)} rows.")

try:
    print("Connecting to database...")
    conn = get_db_connection()
    cur = conn.cursor()
    print("Connected.")
    
    updated_count = 0
    not_found_count = 0
    errors = 0
    total_rows = len(df)

    print("Starting ACEC Chapter update...")

    for index, row in df.iterrows():
        if index % 10 == 0:
            print(f"Processing row {index}/{total_rows}...", flush=True)

        try:
            company_name = row.get('name')
            if pd.isna(company_name) or not str(company_name).strip():
                print(f"Skipping row {index}: Empty company name")
                continue

            # Check if company exists
            # print(f"Checking company: {company_name}")
            cur.execute("SELECT id FROM company WHERE company_name = %s", (company_name,))
            res = cur.fetchone()
            
            if res:
                company_id = res[0]
                
                # Get ACEC Chapter
                chapter = row.get('chapter', '')
                
                # Ensure values are strings (handle NaN) and set to "Non-Chapter" if empty
                if pd.isna(chapter) or not str(chapter).strip():
                    chapter = 'Non-Chapter'
                else:
                    chapter = str(chapter).strip()
                
                # Update company record
                update_query = """
                    UPDATE company 
                    SET acec_chapter = %s, updated_at = CURRENT_TIMESTAMP
                    WHERE id = %s
                """
                cur.execute(update_query, (chapter, company_id))
                
                updated_count += 1
                # print(f"Updated: {company_name}")
            else:
                # print(f"Not found: {company_name}")
                not_found_count += 1
            
        except Exception as e:
            print(f"Error processing row {index}: {e}")
            errors += 1
            conn.rollback()

    conn.commit()
    print(f"Finished. Updated: {updated_count}, Not Found: {not_found_count}, Errors: {errors}")

except Exception as e:
    print(f"Critical Error: {e}")
finally:
    if 'cur' in locals(): cur.close()
    if 'conn' in locals(): conn.close()

